Customer Analysis
Ranks customers by sales volume and profit contribution, using a user-adjustable percentile threshold to identify top contributors for promotions. Excludes generic/non-identifiable cash-sale accounts.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery"

Load analysis silver

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/analysis/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])

sales_only = analysis_silver[analysis_silver["entryType"] == "Sale"].copy()
identifiable = sales_only[sales_only["is_identifiable_customer"]].copy()

print(f"Total Sale rows: {len(sales_only)}")
print(f"Identifiable customer rows: {len(identifiable)}")

Build customer summary

In [0]:
identifiable["profit"] = identifiable["salesAmountActual"] + identifiable["costAmountActual"]
identifiable["net_units"] = -identifiable["quantity"]

identifiable_sorted = identifiable.sort_values("posting_date")

customer_summary = identifiable_sorted.groupby("resolved_customer_no").agg(
    customer_name=("resolved_customer_name", "last"),        # most recent name on record
    customer_address=("resolved_customer_address", "last"),  # most recent address on record
    customer_phone1=("resolved_customer_phone1", "last"),
    customer_phone2=("resolved_customer_phone2", "last"),
    customer_email=("resolved_customer_email", "last"),
    total_sales=("salesAmountActual", "sum"),
    total_profit=("profit", "sum"),
    total_units=("net_units", "sum"),
    order_count=("posting_date", "count"),
    first_purchase=("posting_date", "min"),
    last_purchase=("posting_date", "max"),
).reset_index()

customer_summary = customer_summary[customer_summary["total_units"] > 0].copy()
print(customer_summary.shape)
customer_summary.head()

Compute percentile ranks

In [0]:
customer_summary["sales_percentile"] = customer_summary["total_sales"].rank(pct=True) * 100
customer_summary["profit_percentile"] = customer_summary["total_profit"].rank(pct=True) * 100
customer_summary["volume_percentile"] = customer_summary["total_units"].rank(pct=True) * 100

customer_summary = customer_summary.sort_values("sales_percentile", ascending=False)
print(customer_summary[["customer_name", "total_sales", "sales_percentile", "total_profit", "profit_percentile"]].head(10))

Save the full ranked table

In [0]:
buffer = io.BytesIO()
customer_summary.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/analysis/customer_ranking.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved customer_ranking.xlsx with precomputed percentiles")

from src.io.storage import save_gold
save_gold(blob_service, customer_summary, f"{ANALYSIS_BASE}/analysis/customer_ranking.parquet")

In [0]:
dbutils.widgets.text("sales_percentile_threshold", "90")
sales_threshold = float(dbutils.widgets.get("sales_percentile_threshold"))

top_customers = customer_summary[customer_summary["sales_percentile"] >= sales_threshold]
print(f"Customers at or above the {sales_threshold}th percentile: {len(top_customers)}")
print(top_customers[["customer_name", "customer_address", "customer_phone1", "total_sales", "sales_percentile"]])